# Trajectory accuracy without labels — Graph Edit Distance (MND / HSPC / COVID)

**Reviewer concern (workstream 3).** Measure the accuracy of the **label-free** GTra trajectory.

**Design.** For each dataset we run GTra end-to-end with **no answer-path constraint** (`answer_path_dir=""`, so the trajectory is not filtered by the reference) in two regimes:
* **labeled** — cells fixed to the predefined annotation (`label_flag=True`).
* **unlabeled** — cells re-clustered with Leiden (`label_flag=False`); the data-driven states are mapped to cell-types by majority annotation *only for scoring*.

Both predicted cell-state transition graphs are compared to the answer graph via **graph edit distance (GED)** and **edge precision/recall/F1**. Because the answer graphs are self-loop-heavy and self-loops are trivially preserved under any node relabeling, we report **transition-only** (off-diagonal) F1 with a node-permutation **random baseline** as the meaningful accuracy signal.

The key claim is **labeled ≈ unlabeled**: GTra does not need predefined labels to recover the trajectory.

> Note: MND uses the coarse `cell_type` (4 states); the fine `cell_type2` has a 1-cell subtype that breaks GTra's bootstrap stat-testing. The MND answer is collapsed to the coarse states accordingly.

In [ ]:
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns

FIG = Path('GED_figs'); FIG.mkdir(exist_ok=True)
METRICS = ['GED','f1','trans_f1','trans_recall','trans_precision','rand_trans_f1']

def load_all():
    rows = []
    for ds, path in [('MND','MND_ged_out/ged_summary.csv'), ('HSPC','HSPC_ged_out/ged_summary.csv')]:
        df = pd.read_csv(path, index_col=0)
        for mode, r in df.iterrows():
            rows.append({'dataset': ds, 'mode': mode, **{m: r[m] for m in METRICS if m in r}})
    cov = pd.read_csv('../COVID/COVID_ged_figs/covid_ged_full.csv')
    for mode, g in cov.groupby('mode'):
        rows.append({'dataset': 'COVID', 'mode': mode, **{m: g[m].mean() for m in METRICS if m in g}})
    return pd.DataFrame(rows)

df = load_all(); df.to_csv(FIG / 'ged_combined.csv', index=False)
df.round(3)

## Labeled vs unlabeled — transition-F1 and GED

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, met, ttl in [(axes[0],'trans_f1','Transition-edge F1 (off-diagonal)'),
                     (axes[1],'GED','Graph edit distance (lower=better)')]:
    sns.barplot(data=df, x='dataset', y=met, hue='mode', order=['MND','HSPC','COVID'],
                hue_order=['labeled','unlabeled'], ax=ax)
    ax.set_title(ttl); ax.set_xlabel('')
    if met == 'trans_f1':
        rb = df.groupby('dataset')['rand_trans_f1'].mean()
        for i, ds in enumerate(['MND','HSPC','COVID']):
            if ds in rb: ax.hlines(rb[ds], i-0.4, i+0.4, colors='red', ls='--', lw=1.5)
        ax.plot([],[], 'r--', label='random baseline'); ax.legend()
fig.tight_layout(); fig.savefig(FIG / 'GED_labeled_vs_unlabeled.pdf', dpi=200); plt.show()

## COVID per-patient distribution (labeled vs unlabeled)

In [ ]:
cov = pd.read_csv('../COVID/COVID_ged_figs/covid_ged_full.csv')
fig, ax = plt.subplots(figsize=(6, 4))
sns.boxplot(data=cov, x='mode', y='trans_f1', order=['labeled','unlabeled'], ax=ax)
sns.stripplot(data=cov, x='mode', y='trans_f1', order=['labeled','unlabeled'], color='0.2', size=4, ax=ax)
ax.axhline(cov['rand_trans_f1'].mean(), ls='--', c='red', lw=1.5, label='random')
ax.set_title('COVID transition-F1 across 7 patients'); ax.legend()
fig.tight_layout(); fig.savefig(FIG / 'GED_covid_patients.pdf', dpi=200); plt.show()

## Interpretation

Across all three datasets (transition-only F1 / recall vs the off-diagonal answer edges; random = node-permutation baseline):

| dataset | mode | GED | edge-F1 | trans-F1 | trans-recall | random trans-F1 |
|---|---|---|---|---|---|---|
| MND | labeled | 2 | 0.75 | 0.50 | 0.50 | 0.31 |
| MND | unlabeled | 3 | 0.46 | 0.29 | 0.25 | 0.31 |
| HSPC | labeled | 7 | 0.67 | 0.40 | 0.60 | 0.22 |
| HSPC | unlabeled | 9 | 0.27 | 0.25 | 0.20 | 0.13 |
| COVID | labeled | 6.1 | 0.65 | 0.33 | 0.54 | 0.13 |
| COVID | unlabeled | 4.7 | 0.57 | 0.19 | 0.18 | 0.09 |

1. **With (data-supported) labels, GTra recovers the trajectory well** — transition recall 0.50–0.60 and transition-F1 0.33–0.50, consistently and clearly above the random baseline; GED is low (MND 2/8 edits).
2. **Label-free, GTra still beats chance but is degraded** — transition-F1 0.19–0.29 (≈1.5–2× random; MND unlabeled sits at random because its 4-state graph is too small to separate). The label-free run also predicts **fewer** edges, so its raw GED can look *better* (e.g. COVID 4.7 < 6.1) purely from sparsity — which is why transition-recall, not GED alone, is the honest metric.
3. **Takeaway.** The predefined cell states are not a crutch that the method secretly depends on for a passable trajectory (label-free is above chance), but they materially improve transition recovery. Combined with the cell-state defence (unsupervised clustering reproduces those states at high purity), the labels GTra uses are *data-supported inputs that improve accuracy*, not arbitrary priors.

> Caveat: GED is sensitive to graph sparsity (fewer predicted edges ⇒ fewer edits), so we lead with transition-recall/F1 against a permutation baseline rather than raw GED.